In [1]:
import pandas as pd

In [7]:
train_df = pd.read_parquet('../data/train.parquet')
test_df = pd.read_parquet('../data/test.parquet')

In [5]:
train_df.head()

,date,gold_usd_per_oz,platinum_usd_per_oz,gold_usd_per_oz_return,platinum_usd_per_oz_return,us_fed_funds,us_5y_yield,vix,broad_usd_index,iron_ore_usd_per_tonne,...,sa_yoy_inflation,usd_zar,usd_zar_28,usd_zar_1w_return,usd_zar_1m_return,usd_zar_3m_return,usd_zar_1m_volatility,richards_bay_coal_usd,sa_5y_cds_bp,sa_5y_yield
0,2008-10-10,855.400024,996.700012,-0.019289,-0.007625,0.79,2.77,69.95,97.9990,60.8,...,10.565111,9.3626,10.0284,0.085265,0.066904,-0.004945,0.043376,112.40,455.40,9.115
1,2008-10-13,838.900024,989.099976,-0.019289,-0.007625,0.79,2.95,54.99,97.9686,60.8,...,10.565111,9.1667,9.9890,0.085265,0.066904,-0.004945,0.043376,114.25,455.40,9.130
2,2008-10-14,836.299988,1035.099976,-0.003099,0.046507,1.10,3.01,55.13,96.2544,60.8,...,10.565111,9.0639,10.1693,0.085265,0.066904,-0.004945,0.043376,116.00,250.45,9.110
3,2008-10-15,835.500000,966.700012,-0.000957,-0.066081,1.04,2.90,69.25,97.2045,60.8,...,10.565111,9.2213,10.2797,0.085265,0.066904,-0.004945,0.043376,115.75,369.90,9.165
4,2008-10-16,801.500000,882.799988,-0.040694,-0.086790,0.83,2.84,67.61,98.3680,60.8,...,10.565111,10.2475,10.4116,0.085265,0.066904,-0.004945,0.043376,110.25,419.90,9.505


In [16]:
train_df.drop(columns=['date'], inplace=True)
test_df.drop(columns=['date'], inplace=True)

In [17]:
train_df.dtypes

gold_usd_per_oz               float64
platinum_usd_per_oz           float64
gold_usd_per_oz_return        float64
platinum_usd_per_oz_return    float64
us_fed_funds                  float64
us_5y_yield                   float64
vix                           float64
broad_usd_index               float64
iron_ore_usd_per_tonne        float64
brent_usd_per_barrel          float64
sa_repo_rate                  float64
sa_real_gdp                   float64
sa_cpi                        float64
sa_yoy_inflation              float64
usd_zar                       float64
usd_zar_28                    float64
usd_zar_1w_return             float64
usd_zar_1m_return             float64
usd_zar_3m_return             float64
usd_zar_1m_volatility         float64
richards_bay_coal_usd         float64
sa_5y_cds_bp                  float64
sa_5y_yield                   float64
dtype: object

In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import AdaBoostRegressor

# ---- 1. Set up data ----
target_col = 'usd_zar_28'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_val = test_df.drop(columns=[target_col])
y_val = test_df[target_col]

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = {}

# ---- 2. XGBoost ----
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_val)
results['XGBoost'] = rmse(y_val, xgb_pred)

# ---- 3. LightGBM ----
lgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
lgbm_model.fit(X_train, y_train)
lgbm_pred = lgbm_model.predict(X_val)
results['LightGBM'] = rmse(y_val, lgbm_pred)

# ---- 4. AdaBoost ----
ada_model = AdaBoostRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)
ada_model.fit(X_train, y_train)
ada_pred = ada_model.predict(X_val)
results['AdaBoost'] = rmse(y_val, ada_pred)

# ---- 5. Print results ----
print("RMSE by model:")
for model_name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"  {model_name:10s}: {score:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019282 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4630
[LightGBM] [Info] Number of data points in the train set: 3190, number of used features: 22
[LightGBM] [Info] Start training from score 11.396034
RMSE by model:
  AdaBoost  : 1.8843
  LightGBM  : 2.0071
  XGBoost   : 2.3476
